In [1]:
import pandas as pd

original_csv = 'products.csv'
df = pd.read_csv(original_csv)

selected_columns = ['SPU', '状态', '二级分类']
df_selected = df[selected_columns]

In [5]:
# 列出‘状态’中所有唯一值
unique_statuses = df_selected['状态'].unique()
unique_statuses

array(['开发中 - 抽审待定', '开发中 - 自动通过', '开发中 - 已通过', '开发中 - 待重审', '开发中 - 重审通过',
       '开发中 - 待审核', '开发中 - 待修改', '开发中 - 开发中', '已作废', '正常', '开发中 - 待终审',
       '开发中 - 终审通过', '维护下架', '停产', '疑似侵权', '停售', '开发中 - 已作废'],
      dtype=object)

In [6]:
invalid_statuses = ['已作废', '维护下架', '停产', '疑似侵权', '停售', '开发中 - 已作废']
df_selected = df_selected[~df_selected['状态'].isin(invalid_statuses)]

In [7]:
# 列出‘二级分类’中所有唯一值
unique_secondary_categories = df_selected['二级分类'].unique()
unique_secondary_categories

array(['20574 - 长凳（椅子）', '20559 - 沙发（椅子）', '20535 - 咖啡桌（桌子）',
       '20569 - 玄关桌（桌子）', '200316 - 鞋架', '20558 - 边几（桌子）',
       '20566 - 电视柜和娱乐中心（柜子）'], dtype=object)

In [ ]:
category_mapping = {
    '20574 - 长凳（椅子）': '114 - Benches',
    '20559 - 沙发（椅子）': '83 - Sofa',
    '20535 - 咖啡桌（桌子）': '90 - Coffee Tables',
    '20569 - 玄关桌（桌子）': '104 - Console Tables',
    '200316 - 鞋架': '302 - Shoe Storage',
    '20558 - 边几（桌子）': '91 - End & Side Tables',
    '20566 - 电视柜和娱乐中心（柜子）': '88 - TV Stands & Entertainment Centers'
}

df_selected['Mapped Category'] = df_selected['二级分类'].map(category_mapping)


In [9]:
len(df_selected)

73503

In [ ]:
df_selected.to_csv('processed_products.csv', index=False)

In [10]:
# 各个类别的spu数量
df_selected['Mapped Category'].value_counts()

Mapped Category
90 - Coffee Tables                        25025
88 - TV Stands & Entertainment Centers    11543
104 - Console Tables                      10595
83 - Sofa                                  9327
91 - End & Side Tables                     8552
114 - Benches                              8021
302 - Shoe Storage                          440
Name: count, dtype: int64

In [11]:
# 总共选5000个，按比例计算每个类别需要多少个
category_counts = df_selected['Mapped Category'].value_counts()
total_count = category_counts.sum()
desired_total = 5000
category_sample_counts = (category_counts / total_count * desired_total).round().astype(int)
category_sample_counts

Mapped Category
90 - Coffee Tables                        1702
88 - TV Stands & Entertainment Centers     785
104 - Console Tables                       721
83 - Sofa                                  634
91 - End & Side Tables                     582
114 - Benches                              546
302 - Shoe Storage                          30
Name: count, dtype: int64

In [ ]:
import aiohttp
import asyncio
import numpy as np

base_url = 'https://erp.baycheer.com/'

session = aiohttp.ClientSession(
    connector=aiohttp.TCPConnector(limit=100),
    timeout=aiohttp.ClientTimeout(total=30),
    headers={
        "Content-Type": "application/json",
        "app-id": '392028',
        "app-token": 'WRFDN0O98N7QP91HMTVHL9HE58G32UCD'
    }
)

async def fetch_image_vecs(session, spus:str):
    url = base_url + 'api/product/getProductImageVecs'
    payload = {"spus": spus}
    async with session.post(url, json=payload) as response:
        if response.status == 200:
            return await response.json()
        else:
            print(f"Error: {response.status}")
            return None
    

spus = '21779509'
response = asyncio.run(fetch_image_vecs(session, spus))
response

